## TASK 1 — First Calls + Sampling Dial

In [8]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

MODEL = "gemini-2.5-flash"

In [20]:
def call_llm(prompt):
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={"temperature": 0.1}
        )

        time.sleep(1.5)  # API slow down
        return response.text.strip()

    except Exception as e:
        print("Error:", e)
        time.sleep(2)
        return "ERROR"

In [9]:
#2. First working call (response + token usage)
response = client.models.generate_content(
    model=MODEL,
    contents=[
        "You are a concise support assistant.",
        "Explain what tokenization is in one sentence."
    ]
)

print("=== RESPONSE ===")
print(response.text)

print("\n=== TOKEN USAGE ===")
print(response.usage_metadata)

=== RESPONSE ===
Tokenization is the process of breaking down a stream of text into smaller units called tokens.

=== TOKEN USAGE ===
cache_tokens_details=None cached_content_token_count=None candidates_token_count=18 candidates_tokens_details=None prompt_token_count=17 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=17
)] thoughts_token_count=29 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=64 traffic_type=None


In [11]:
import time

for i in range(3):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0.1}
    )

    print(response.text)
    time.sleep(2)  

A support ticket is a tracked request for assistance submitted to a support team to resolve an issue, answer a question, or fulfill a service need.
A support ticket is a tracked request for assistance with a problem or question, submitted to a support team.
A support ticket is a tracked request for assistance submitted to a support team regarding a problem, question, or service need.


In [12]:
#4. Temperature = 1.0 (3 runs)
print("TEMPERATURE = 1.0\n")

for i in range(3):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 1.0
        }
    )

    print(f"Run {i+1}: {response.text}")

TEMPERATURE = 1.0

Run 1: A support ticket is a formal request for assistance submitted by a user to a support team to track and resolve an issue or question.
Run 2: A support ticket is a formal, tracked request from a user seeking assistance or resolution from a support team regarding an issue, question, or problem with a product or service.
Run 3: A support ticket is a formal request for assistance to resolve an issue or answer a question regarding a product or service.


### Observation — What changed and why

When the temperature was set to 0.1, the model produced very similar and stable outputs across all runs. The responses were almost identical because the model mostly selected the highest-probability tokens at each step. This makes the generation more deterministic and focused.

When the temperature was increased to 1.0, the outputs became more diverse and varied in wording and structure. The model started sampling from a wider range of possible next tokens, including less likely ones, which introduced creativity and variation in the responses.

This behavior is directly related to the sampling mechanism explained in the lesson: each step in generation produces a probability distribution over possible next tokens. Temperature controls how that distribution is sampled—low temperature sharpens the distribution (favoring the most likely token), while high temperature flattens it (allowing more randomness and diversity).

## Task 2 — Prompt-pattern bake-off

In [13]:
tickets = [
    "I was charged twice this month",
    "The app crashes when I upload a photo",
    "Please add dark mode",
    "I cannot log in to my account",
    "Payment page is not loading",
    "Can you add export to PDF?",
    "I want a refund for my purchase",
    "The app is very slow lately"
]

In [14]:
def zero_shot(ticket):
    prompt = f"""
Classify the support ticket into one of the following categories:

- billing
- bug
- feature_request
- other

Ticket:
{ticket}

Return only the label.
"""
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0.1}
    )
    return response.text.strip()

In [15]:
def few_shot(ticket):
    prompt = f"""
Examples:

Ticket: I want a refund
Label: billing

Ticket: The app crashes on startup
Label: bug

Ticket: Add dark mode
Label: feature_request

Now classify the following ticket:

Ticket: {ticket}
Label:
"""
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0.1}
    )
    return response.text.strip()

In [16]:
def cot(ticket):
    prompt = f"""
You are a support ticket classifier.

Think briefly before answering:
1. What is the user trying to do or report?
2. Which category fits best?

Categories:
- billing
- bug
- feature_request
- other

Ticket:
{ticket}

Return format:
Reason: <short reason>
Label: <final label only>
"""
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0.1}
    )
    return response.text.strip()

In [24]:
results = []

for i, t in enumerate(tickets):

    print(f"Processing ticket {i+1}/{len(tickets)}")

    results.append({
        "ticket": t,
        "zero_shot": call_llm(zero_shot(t)),
        "few_shot": call_llm(few_shot(t)),
        "cot": call_llm(cot(t))
    })

    time.sleep(3)  # API slow-down

Processing ticket 1/8


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 56.555874139s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},